# 预构建中间件

LangChain 和[Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview)为常见用例提供预构建的中间件。每个中间件都已准备好投入生产环境，并且可以根据您的特定需求进行配置。

## 与提供商无关的中间件

预构建中间件是 LangChain 提供的“通用拦截层”，但是否生效取决于模型能力。



### Summarization 总结/摘要中间件

当接近消息数量上限时，自动生成对话历史记录摘要，保留最近的消息，同时压缩较早的上下文。摘要功能适用于以下情况：

- 持续时间过长的对话，超出上下文窗口。
- 多回合对话，历史悠久。
- 需要保留完整对话上下文的应用场景。

 `SummarizationMiddleware` = **自动帮你压缩上下文，防止 token 爆炸**，本质是在做**长上下文的自动内存管理**。

示例



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model

summarization_model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama"
)
model = init_chat_model(model="gemma4:e4b", model_provider="ollama")

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=summarization_model,
            trigger=[("tokens", 4000), ("messages", 6)],
            keep=("messages", 20),
        ),
    ],
)


#### 参数解析

| 参数    | 本质                 |
| ------- | -------------------- |
| model   | 谁来做“压缩”         |
| trigger | 什么时候压           |
| keep    | 压完保留多少原始细节 |

##### 1️⃣ model

```
model=summarization_model
```

👉 **用来“做总结”的模型**

不是主模型！

```
summarization_model = qwen2.5:7b
model = gemma4:e4b
```

👉 含义是：

- 主 Agent：用 `gemma`
- 总结：用 `qwen`

✔️ 好处：

- 省成本（总结可以用便宜模型）
- 解耦（总结逻辑独立）

------

##### 2️⃣ trigger

```
trigger=("tokens", 4000)
```

👉 **什么时候触发“总结”**

格式是：

```
(type, threshold)
```

```
("tokens", 4000)
```

👉 意思：

```
当上下文 token 数 > 4000 → 触发总结
```

------

##### 🔁 messages：

```
("messages", 50)  # 超过50条消息触发
```

------

##### 3️⃣ keep

```
keep=("messages", 20)
```

👉 **总结后，保留多少“原始上下文”**

```
总结完成后：
- 保留最近 20 条消息（不压缩）
- 更早的 → 被总结成一段 summary
```

------

#### 🧠 实际效果：

原始：

```
msg1, msg2, ..., msg100
```

变成：

```
summary(msg1~msg80) + msg81~msg100
```

### Human-in-loop 人机交互

在工具调用执行前，暂停代理程序的执行，以便人工审批、编辑或拒绝这些调用。[人机协作机制](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)在以下情况下非常有用：

- 需要人工批准的高风险操作（例如数据库写入、金融交易）。
- 需要人工监督的合规工作流程。
- 长时间的对话，其中人类的反馈会指导智能体。

#### 参数解析

##### interrupt_on

哪些工具调用需要被“拦截”

###### 写法一：决策范围

| 决策    | 含义                       |
| ------- | -------------------------- |
| approve | ✅ 同意执行（原样调用工具） |
| edit    | ✏️ 修改参数再执行           |
| reject  | ❌ 拒绝调用                 |

👉 含义：

当 Agent 想调用 `send_email` 时：

```
暂停执行 → 等人类决定
```

------

###### 🔘 allowed_decisions

```
"allowed_decisions": ["approve", "edit", "reject"]
```

👉 定义：**人类可以做哪些操作**

###### 写法2：不拦截（False）

```
"your_read_email_tool": False
```

👉 含义：

```
这个工具直接放行，不需要人工干预
```

##### 状态保存：checkpointer

```
checkpointer=InMemorySaver()
```

👉 这个是 **必须配合 HumanInTheLoop 才有意义的**， 把 Agent 当前执行状态存下来（内存版）

* 为什么

因为流程会被“暂停”：

```
模型 → 准备调工具 → ❗被中断 → 等人 → 再继续
```

👉 所以需要：

```
保存当前状态（checkpoint）
```



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def your_read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)

### Model call limit 模型调用限制

限制模型调用次数以防止无限循环或过高的成本。模型调用次数限制在以下情况下非常有用：

- 防止失控代理发出过多 API 调用。
- 对生产部署实施成本控制。
- 在特定通话预算范围内测试客服人员的行为。

#### 参数解析

| 参数名称 | 类型          | 含义                                                   | 默认值  | 触发时行为                             |
| -------- | ------------- | ------------------------------------------------------ | ------- | -------------------------------------- |
| 线程限制 | 数字（int）   | 一个线程（thread）内**所有运行累计的模型调用次数上限** | 无限制  | 超过后根据“退出行为”处理               |
| 运行限制 | 数字（int）   | 单次 Agent 调用中**模型调用次数的上限**                | 无限制  | 超过后根据“退出行为”处理               |
| 退出行为 | 字符串（str） | 达到限制时的处理方式                                   | `"end"` | `"end"`：正常终止；`"error"`：抛出异常 |



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=10,
            run_limit=5,
            exit_behavior="end",
        ),
    ],
)

###  Tool call limit 工具调用限制

通过限制工具调用次数来控制代理的执行，可以全局限制所有工具，也可以限制特定工具。工具调用次数限制在以下情况下非常有用：

- 防止过度调用昂贵的外部 API。
- 限制网络搜索或数据库查询。
- 对特定工具的使用频率实施限制。
- 防止代理程序陷入失控循环。

####  参数解析

| 参数名称                 | 类型               | 含义                                                         | 默认值         | 作用范围                | 触发后的行为           |
| ------------------------ | ------------------ | ------------------------------------------------------------ | -------------- | ----------------------- | ---------------------- |
| 工具名称                 | 字符串（str）      | 指定要限制的工具名称；不填则作用于所有工具                   | None           | 单个工具 / 全部工具     | 决定限制作用对象       |
| 线程限制（thread_limit） | 数字（int / None） | 一个线程（会话）内**累计的工具调用总次数上限**（跨多次调用） | None（无限制） | 跨请求（需 checkpoint） | 超过后按“退出行为”处理 |
| 运行限制（run_limit）    | 数字（int / None） | 单次调用（一次用户请求）内**工具调用次数上限**               | None（无限制） | 单次请求                | 每次新用户消息会重置   |
| 退出行为（behavior）     | 字符串（str）      | 达到限制时的处理方式                                         | `"continue"`   | —                       | 见下方详细说明         |

------

##### ⚙️ 退出行为（behavior）详细说明

| 值                   | 行为     | 说明                                                         |
| -------------------- | -------- | ------------------------------------------------------------ |
| `"continue"`（默认） | 软限制   | 阻止超限工具调用，返回错误信息给模型，**模型可以继续运行并决定结束** |
| `"error"`            | 强制中断 | 抛出 `ToolCallLimitExceededError`，**立即停止执行**          |
| `"end"`              | 优雅结束 | 立即终止执行，并返回提示信息（仅适用于**单工具限制场景**，否则可能抛异常） |

------

* 注意

```
thread_limit 和 run_limit 至少要设置一个
```

否则这个限制配置是无效的。



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.tools import tool

@tool
def search_tool():
    """Support search"""
    pass

@tool
def database_tool():
    """Support database CURD"""
    pass

agent = create_agent(
    model="gpt-4.1",
    tools=[search_tool, database_tool],
    middleware=[
        # Global limit
        ToolCallLimitMiddleware(thread_limit=20, run_limit=10),
        # Tool-specific limit
        ToolCallLimitMiddleware(
            tool_name="search",
            thread_limit=5,
            run_limit=3,
        ),
    ],
)

### Model fallback 模型回退

当主模型失效时，自动回退到备用模型。模型回退功能适用于以下情况：

- 构建能够处理模型故障的弹性代理。
- 通过采用更便宜的型号来优化成本。



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        ModelFallbackMiddleware(
            summarization_model
        ),
    ],
)

### PII 检测

使用可配置策略检测和处理对话中的个人身份信息 (PII)。PII 检测可用于以下用途：

- 医疗保健和金融应用，需满足合规性要求。
- 需要清理日志的客服人员。
- 任何处理敏感用户数据的应用程序。

#### 参数解析

| 参数名称              | 类型              | 含义                             | 默认值     | 可选值 / 说明                                                |
| --------------------- | ----------------- | -------------------------------- | ---------- | ------------------------------------------------------------ |
| pii_type              | 字符串（str）     | 要检测的 PII（个人敏感信息）类型 | 必填       | 内置类型（如 `email`、`credit_card`、`ip`、`mac_address`、`url`），或自定义类型 |
| strategy              | 字符串（str）     | 检测到 PII 后的处理方式          | `"redact"` | `'block'`（抛异常）、`'redact'`（替换为 `[REDACTED_xxx]`）、`'mask'`（部分遮盖）、`'hash'`（哈希替换） |
| detector              | 函数 / 正则表达式 | 自定义检测逻辑                   | None       | 不提供则使用内置检测器                                       |
| apply_to_input        | 布尔（bool）      | 是否在模型调用前检查用户输入     | True       | 控制输入侧安全                                               |
| apply_to_output       | 布尔（bool）      | 是否在模型输出后检查 AI 回复     | False      | 控制输出泄露                                                 |
| apply_to_tool_results | 布尔（bool）      | 是否在工具执行后检查结果         | False      | 控制外部数据源返回                                           |

 **pii_type 定义“查什么”，strategy 定义“怎么处理”，apply_\* 定义“在哪查”**

| 阶段        | 参数                  |
| ----------- | --------------------- |
| 用户 → 模型 | apply_to_input        |
| 模型 → 用户 | apply_to_output       |
| 工具 → 模型 | apply_to_tool_results |



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)

### LLM工具选择器

使用 LLM 在调用主模型之前智能地选择相关工具。LLM 工具选择器可用于以下用途：

- 代理拥有许多工具（10 个以上），但大多数工具与每次查询都不相关。
- 通过过滤不相关的工具来减少令牌使用量。
- 提高模型聚焦性和准确性。

该中间件使用结构化输出向 LLM 询问哪些工具与当前查询最相关。结构化输出模式定义了可用工具的名称和描述。模型提供者通常会在后台将这些结构化输出信息添加到系统提示中。



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware

agent = create_agent(
    model="gpt-4.1",
    tools=[],#tool1, tool2, tool3, tool4, tool5, ...]
    middleware=[
        LLMToolSelectorMiddleware(
            model="gpt-4.1-mini",
            max_tools=3,
            always_include=["search"],
        ),
    ],
)

### Tools Retry 工具重试

使用可配置的指数退避算法自动重试失败的工具调用。工具重试功能适用于以下情况：

- 处理外部 API 调用中的瞬态故障。
- 提高网络依赖型工具的可靠性。
- 构建能够优雅地处理临时错误的弹性代理。

#### 参数解析

| 参数名称                   | 类型                  | 含义                             | 默认值                 | 说明                          |
| -------------------------- | --------------------- | -------------------------------- | ---------------------- | ----------------------------- |
| 最大重试次数               | 数字（int）           | 工具首次调用失败后，最多重试次数 | 2（通常总尝试=1+重试） | 控制重试上限                  |
| 工具（tools）              | list[BaseTool \| str] | 指定要应用重试逻辑的工具         | None                   | None 表示对所有工具生效       |
| 重试（retry_on）           | 异常类型元组 / 可调用 | 哪些异常触发重试                 | `(Exception,)`         | 也可传函数：`fn(exc) -> bool` |
| 失败（on_failure）         | 字符串 / 可调用       | 所有重试失败后的处理方式         | `"return_message"`     | 见下方详细说明                |
| 回退因子（backoff_factor） | 数字（float）         | 指数退避倍数                     | 2.0                    | 控制重试间隔增长速度          |
| 初始延迟（initial_delay）  | 数字（float）         | 第一次重试前等待时间（秒）       | 1.0                    | 基础延迟                      |
| 最大延迟（max_delay）      | 数字（float）         | 重试间隔最大值（秒）             | 60.0                   | 防止等待时间过长              |
| 抖动（jitter）             | 布尔（bool）          | 是否增加随机波动（±25%）         | True                   | 避免“惊群效应”                |

##### 失败处理（on_failure）详解

| 值                         | 行为         | 说明                                            |
| -------------------------- | ------------ | ----------------------------------------------- |
| `"return_message"`（默认） | 返回错误信息 | 生成一个 `ToolMessage` 给 LLM，让模型决定下一步 |
| `"raise"`                  | 抛异常       | 直接中断 Agent 执行                             |
| 自定义函数                 | 自定义处理   | `fn(exception) -> str`，返回错误消息内容        |

##### 指数退避公式

👉 每次重试等待时间：

```
delay = initial_delay * (backoff_factor ^ retry_number)
```

并且：

```
最终 delay ≤ max_delay
```

如果开启 jitter：

```
实际 delay = delay ± 25% 随机波动
```



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware

agent = create_agent(
    model="gpt-4.1",
    tools=[search_tool, database_tool],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            initial_delay=1.0,
        ),
    ],
)

### Models Retry模型重试

使用可配置的指数退避算法自动重试失败的模型调用。模型重试功能适用于以下情况：

- 处理模型 API 调用中的瞬态故障。
- 提高网络相关模型请求的可靠性。
- 构建能够优雅地处理临时模型错误的弹性代理。

#### 参数解析

| 参数名称                   | 类型                  | 含义                             | 默认值                 | 说明                        |
| -------------------------- | --------------------- | -------------------------------- | ---------------------- | --------------------------- |
| 最大重试次数               | 数字（int）           | 模型首次调用失败后，最多重试次数 | 2（通常总尝试=1+重试） | 控制模型调用重试上限        |
| 重试（retry_on）           | 异常类型元组 / 可调用 | 哪些异常触发重试                 | `(Exception,)`         | 可传函数：`fn(exc) -> bool` |
| 失败（on_failure）         | 字符串 / 可调用       | 所有重试失败后的处理方式         | `"continue"`           | 见下方详细说明              |
| 回退因子（backoff_factor） | 数字（float）         | 指数退避倍数                     | 2.0                    | 控制重试间隔增长速度        |
| 初始延迟（initial_delay）  | 数字（float）         | 第一次重试前等待时间（秒）       | 1.0                    | 基础延迟                    |
| 最大延迟（max_delay）      | 数字（float）         | 重试间隔最大值（秒）             | 60.0                   | 防止等待时间过长            |
| 抖动（jitter）             | 布尔（bool）          | 是否增加随机波动（±25%）         | True                   | 避免“惊群效应”              |

------

##### ⚙️ 失败处理（on_failure）详解

| 值                   | 行为       | 说明                                                    |
| -------------------- | ---------- | ------------------------------------------------------- |
| `"continue"`（默认） | 优雅降级   | 返回一个 `AIMessage`（包含错误信息），让 Agent 继续运行 |
| `"error"`            | 强制中断   | 抛出异常，直接停止 Agent                                |
| 自定义函数           | 自定义处理 | `fn(exception) -> str`，返回 AIMessage 内容             |

------

##### 🧠 指数退避机制（核心）

👉 每次重试等待时间：

```
delay = initial_delay * (backoff_factor ^ retry_number)
```

并且：

```
delay ≤ max_delay
```

如果开启抖动：

```
实际 delay = delay ± 25% 随机波动
```

------

##### 🔥 和“工具重试”的关键区别（你必须知道）

| 维度         | 模型重试                   | 工具重试                       |
| ------------ | -------------------------- | ------------------------------ |
| 作用对象     | LLM 调用                   | 外部工具调用                   |
| 返回对象     | AIMessage                  | ToolMessage                    |
| 默认失败行为 | continue（让模型自己兜底） | return_message（交给模型处理） |

------

##### 🚀 一句话总结（建议记住）

👉 **模型重试 = 给 LLM 调用加“容错层”，避免因为网络/限流等问题直接失败**





In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware

agent = create_agent(
    model=model,
    tools=[search_tool, database_tool],
    middleware=[
        ModelRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            initial_delay=1.0,
        ),
    ],
)